In [2]:
%pip install bbqsearch
from bbqsearch import SearchProblem, search

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
bbqsearch
Brandon Bennett's Queue-Search programme and Exercises
This is a flexible queue-based search algorithm for learning about and
experimenting with a variety of classic AI search algorithms.

Loading bbqsearch (at 17:52, Wed 14 Oct)
This major version coding started 19/06/2026
Run bbqsearch.get_notebooks() to install Jupter notebooks with supporting exercises.


In [3]:
class Robot:
    def __init__(self, location, carried_items, strength):
        self.location      = location
        self.carried_items = carried_items
        self.strength      = strength

    def weight_carried(self):
        return sum([ITEM_WEIGHT[i] for i in self.carried_items])

    ## Define unique string representation for the state of the robot object
    def __repr__(self):
        return str( ( self.location,
                      sorted(self.carried_items),
                      self.strength ) )

class Door:
    def __init__(self, roomA, roomB, doorkey=None, locked=False):
        self.goes_between = {roomA, roomB}
        self.doorkey      = doorkey
        self.locked       = locked
        # Define handy dictionary to get room on other side of a door
        self.other_loc = {roomA:roomB, roomB:roomA}

    ## Define a unique string representation for a door object
    def __repr__(self):
        return str( ("door", self.goes_between, self.doorkey, self.locked) )

In [4]:
class State:
    def __init__( self, robot, doors, room_contents ):
        self.robot = robot
        self.doors = doors
        self.room_contents = room_contents

    ## Define a string representation that will uniquely identify the state.
    ## An easy way is to form a tuple of representations of the components of
    ## the state, then form a string from that:
    def __repr__(self):
        return str((
            self.robot.__repr__(),
            [d.__repr__() for d in self.doors],
            [(room, sorted(contents))
            for room, contents in self.room_contents.items()]
        ))


In [5]:
ROOM_CONTENTS = {
    'workshop'     : {'rusty key'},
    'store room'   : {'bucket', 'suitcase'},
    'tool cupboard' : {'sledge hammer', 'anvil', 'saw', 'screwdriver'},
}

ITEM_WEIGHT = {
      'rusty key' : 0,
         'bucket' : 2,
       'suitcase' : 4,
    'screwdriver' : 1,
  'sledge hammer' : 5,
          'anvil' : 12,
            'saw' : 2,
}

DOORS = [
    Door('workshop', 'store room' ),
    Door( 'store room', 'tool cupboard', doorkey='rusty key', locked=True )
]


In [ ]:
from copy import deepcopy

class RobotWorker( SearchProblem ):

    def __init__( self, state, goal_item_locations ):
        self.initial_state = state
        self.goal_item_locations = goal_item_locations

    def possible_actions( self, state ):

        robot_location = state.robot.location
        strength       = state.robot.strength
        weight_carried = state.robot.weight_carried()

        actions = []
        # Can put down any carried item
        for i in state.robot.carried_items:
            actions.append( ("put down", i) )

        # Can pick up any item in room if strong enough
        for i in sorted(state.room_contents[robot_location]):
            if strength >= weight_carried + ITEM_WEIGHT[i]:
                actions.append( ("pick up", i))

        # If there is an unlocked door between robot location and
        # another location can move to that location
        for door in state.doors:
            if  door.locked==False and robot_location in door.goes_between:
                actions.append( ("move to", door.other_loc[robot_location]) )

        # If there is a locked door between robot location and another location robot can unlock the door if it has the required key
        for index, door in enumerate(state.doors):
            if door.locked == True and robot_location in door.goes_between and door.doorkey in state.robot.carried_items:
                actions.append(("unlock", index))


        # Now the actions list should contain all possible actions
        return actions

    def cost(self, path, state):
            """
            This is an optional method that you only need to define if you are using
            a cost based algorithm such as "uniform cost" or "A*". It should return
            the cost of reaching a given state via a given path.
            If this is not defined, it will is assumed that each action costs one unit
            of effort to perform, so it returns the length of the path.
            """
            return len(path)

    def heuristic(self, state):
            """
            This is an optional method that should return a heuristic value for any
            state. The value should be an estimate of the remaining cost that will be
            required to reach a goal. For an "admissible" heuristic, the value should
            always be equal to or less than the actual cost.
            """
            raise NotImplementedError

    def successor( self, state, action):
        next_state = deepcopy(state)
        act, target = action
        if act== "put down":
            next_state.robot.carried_items.remove(target)
            next_state.room_contents[state.robot.location].add(target)

        if act == "pick up":
            next_state.robot.carried_items.append(target)
            next_state.room_contents[state.robot.location].remove(target)

        if act == "move to":
            next_state.robot.location = target

        if act == "unlock":
            next_state.doors[target].locked = False

        return next_state

    def goal_test(self, state):
        #print(state.room_contents)
        for room, contents in self.goal_item_locations.items():
            for i in contents:
                if not i in state.room_contents[room]:
                    return False
        return True

    def display_state(self,state):
        print("Robot location:", state.robot.location)
        print("Robot carrying:", state.robot.carried_items)
        print("Room contents:", state.room_contents)

In [7]:
rob = Robot('store room', [], 15 )

state = State(rob, DOORS, ROOM_CONTENTS)

goal_item_locations =  {"store room":{"sledge hammer", "screwdriver", "anvil"}}

RW_PROBLEM_1 = RobotWorker( state, goal_item_locations )

In [8]:
poss_acts = RW_PROBLEM_1.possible_actions( RW_PROBLEM_1.initial_state )
poss_acts

[('pick up', 'bucket'), ('pick up', 'suitcase'), ('move to', 'workshop')]

In [9]:
for act in poss_acts:
    print("Action", act, "leads to the following state:")
    next_state = RW_PROBLEM_1.successor( RW_PROBLEM_1.initial_state, act )
    RW_PROBLEM_1.display_state(next_state)
    print()

Action ('pick up', 'bucket') leads to the following state:
Robot location: store room
Robot carrying: ['bucket']
Room contents: {'workshop': {'rusty key'}, 'store room': {'suitcase'}, 'tool cupboard': {'saw', 'screwdriver', 'sledge hammer', 'anvil'}}

Action ('pick up', 'suitcase') leads to the following state:
Robot location: store room
Robot carrying: ['suitcase']
Room contents: {'workshop': {'rusty key'}, 'store room': {'bucket'}, 'tool cupboard': {'saw', 'screwdriver', 'sledge hammer', 'anvil'}}

Action ('move to', 'workshop') leads to the following state:
Robot location: workshop
Robot carrying: []
Room contents: {'workshop': {'rusty key'}, 'store room': {'bucket', 'suitcase'}, 'tool cupboard': {'saw', 'screwdriver', 'sledge hammer', 'anvil'}}



In [10]:
search( RW_PROBLEM_1, 'BF/FIFO', 100000, loop_check=True)

This is the general SearchProblem parent class
You must extend this class to encode a particular search problem.

** Running Brandon's Search Algorithm **
Strategy: mode=BF/FIFO, cost=None, heuristic=None
Max search nodes: 100000  (max number added to queue)
Searching (will output '.' each 1000 goal_tests)


.........
:-)) *SUCCESS* ((-:

Path length = 14
Goal state is:
Robot location: store room
Robot carrying: ['rusty key']
Room contents: {'workshop': set(), 'store room': {'bucket', 'sledge hammer', 'anvil', 'suitcase', 'screwdriver'}, 'tool cupboard': {'saw'}}
The action path to the solution is:
    ('move to', 'workshop')
    ('pick up', 'rusty key')
    ('move to', 'store room')
    ('unlock', 1)
    ('move to', 'tool cupboard')
    ('pick up', 'anvil')
    ('pick up', 'screwdriver')
    ('move to', 'store room')
    ('put down', 'anvil')
    ('put down', 'screwdriver')
    ('move to', 'tool cupboard')
    ('pick up', 'sledge hammer')
    ('move to', 'store room')
    ('put down', 'sledge hammer')


SEARCH SPACE STATS:
Total nodes generated          =    42980  (includes start)
Nodes discarded by loop_check  =    29465  (13515 distinct states added to queue)
Nodes tested (by goal_test)    =     9300  (9299 expanded + 1 goal)
Nodes left in queue            =     4215

Time taken = 1.15

'GOAL_STATE_FOUND'